In [1]:
import json
import random
import torch
import pandas as pd
import numpy as np

import pathlib, pyrootutils

PROJECT_ROOT = pyrootutils.find_root(search_from=pathlib.Path.cwd(), indicator=".project-root")
pyrootutils.setup_root(PROJECT_ROOT, pythonpath=True)
DATA_DIR = PROJECT_ROOT / "data"

In [2]:
from rag.rag import embed_texts, chunk
from main import generate_batchfile

/Users/jauliegoe/Desktop/NYU MSDS/SCFG/llm-scfg/scfg/utils.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [5]:
file_path = PROJECT_ROOT / "batches/responses_d5a3588489c8e427_o4-mini.jsonl"

In [ ]:
# Parse the RAG responses JSONL file and extract key information.
responses = []

with open(file_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            
            # Extract request metadata
            request_metadata = data['request']['body']['metadata']
            
            # Extract response info
            response_data = data['response']
            choice = response_data['choices'][0]
            usage = response_data['usage']
            
            # Create response record
            response_record = {
                'custom_id': data['custom_id'],
                'input_sentence': request_metadata['input_sentence'],
                'expected_output': request_metadata['output_sentence'],
                'grammar_name': request_metadata['grammar_name'],
                'depth': int(request_metadata['depth']),
                'model': request_metadata['model'],
                'n_words': int(request_metadata['n_words']),
                'n_rules': int(request_metadata['n_rules']),
                
                # Response analysis
                'response_content': choice['message']['content'],
                'finish_reason': choice['finish_reason'],
                'completion_tokens': usage['completion_tokens'],
                'prompt_tokens': usage['prompt_tokens'],
                'total_tokens': usage['total_tokens'],
                
            # Check if response is empty
            'is_empty': len(choice['message']['content'].strip()) == 0,
            'has_final_answer': 'Final answer:' in choice['message']['content'],
            
            # Extract final answer if present
            'final_answer': choice['message']['content'].split('Final answer:')[-1].strip() if 'Final answer:' in choice['message']['content'] else None,
            
            # Check if reasoning is present
            'has_reasoning': len(choice['message']['content']) > 0 and 'Final answer:' not in choice['message']['content'][:100]
            }
            
            responses.append(response_record)

In [11]:
responses[5]

{'custom_id': 'request-5',
 'input_sentence': 'sotmixsoq kacraznitzoxwek luphattuf sotmixsoq yefvaxfinficpof luphattuf cefjahdew topyal cotxujduq vazmogley',
 'expected_output': 'kojxok kojxok wazyesnipjiqjikgeb ruxfugzed vubyuz xiqkajfaw dannov rigtumkahyovsadxur joqqexsudwoc rigtumkahyovsadxur',
 'grammar_name': 'd5a3588489c8e427',
 'depth': 2,
 'model': 'o4-mini',
 'n_words': 46,
 'n_rules': 26,
 'response_content': 'I translated each input word by looking up its first‐language form in the lex rules and replacing it with the corresponding second‐language form; words not in any lex rule I left unchanged.\n\nsotmixsoq  →  rigtumkahyovsadxur  \nkacraznitzoxwek  →  joqqexsudwoc  \nluphattuf  →  luphattuf  \nsotmixsoq  →  rigtumkahyovsadxur  \nyefvaxfinficpof  →  dannov  \nluphattuf  →  luphattuf  \ncefjahdew  →  xiqkajfaw  \ntopyal  →  vubyuz  \ncotxujduq  →  cotxujduq  \nvazmogley  →  ruxfugzed  \n\nFinal answer: rigtumkahyovsadxur joqqexsudwoc luphattuf rigtumkahyovsadxur dannov lupha

In [ ]:
# TODO: Need to create a deeper samples file & run RAG experiment
# Probably need to organize the way the rules are chunked